# Coding Assignment 2: Solving Linear Systems: LU, Inverses, and Determinants

---

## 🎯 What You'll Learn

By the end of this notebook you will be able to:

- Relate the course convention $P_{\rm course}A=LU$ to SciPy's convention $A=P_{\rm scipy}LU$, then solve $Ax=b$ by forward and back substitution
- Explain why `np.linalg.solve(A, b)` is preferred over `inv(A) @ b`, and time the difference
- Compute determinants stably from an LU factorization, and test invertibility
- Fit a polynomial regression through the normal equations, and read its condition number
- Continue your **mini-transformer milestone**: diagnose a rank-deficient linear projection and interpret its non-unique weights

> **Prerequisite:** This notebook assumes you completed **Coding HW 1**, which covered the Python
> and NumPy warm-up and built the embedding matrix `E` and the lookup `X = embed_tokens(...)`.
> Part 3 here rebuilds `X` in two lines and moves straight on to the linear algebra.

## ⏱ Estimated Time: 2–3 hours

## 📦 What to Submit

Upload this completed `.ipynb` file to the course portal. All `# TODO` cells must be filled in, and all assertion cells must pass without errors.

---


## Setup

Run this cell first — it imports everything you need for this assignment.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import lu

# Our shared mini-transformer utilities (grows each week)
import sys
sys.path.insert(0, ".")
import mt_utils

np.set_printoptions(precision=4, suppress=True)
print("Setup complete. NumPy version:", np.__version__)


Setup complete. NumPy version: 2.5.5


---

## Part 1: Linear Algebra in Python — LU Factorization and Inverses

### 1.1 LU Factorization with SciPy

Recall from lecture: Gaussian elimination with row swaps produces $P_{\rm course}A=LU$, where:
- $P$ is a permutation matrix (handles row swaps)
- $L$ is lower triangular with 1s on the diagonal
- $U$ is upper triangular

**Why factor?** Once you have $LU$, solving $Ax = b$ costs only $O(n^2)$ per right-hand side — far cheaper than recomputing from scratch.

SciPy provides `scipy.linalg.lu(A)` but uses $A=P_{\rm scipy}LU$. Therefore $P_{\rm course}=P_{\rm scipy}^{\top}$. The code below verifies both equivalent statements:


In [2]:
# 1.1 LU Factorization
A = np.array([[2., 1., 1.],
              [4., 3., 3.],
              [8., 7., 9.]])

P, L, U = lu(A)

print("P_scipy (permutation):\n", P)
print("L (lower triangular):\n", L.round(4))
print("U (upper triangular):\n", U.round(4))

# SciPy: A = P_scipy @ L @ U.
# Course notation: P_course @ A = L @ U, where P_course = P_scipy.T.
print("\nSciPy check, A = P_scipy L U:", np.allclose(A, P @ L @ U))
print("Course check, P_course A = L U:", np.allclose(P.T @ A, L @ U))


P_scipy (permutation):
 [[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
L (lower triangular):
 [[1.     0.     0.    ]
 [0.25   1.     0.    ]
 [0.5    0.6667 1.    ]]
U (upper triangular):
 [[ 8.      7.      9.    ]
 [ 0.     -0.75   -1.25  ]
 [ 0.      0.     -0.6667]]

SciPy check, A = P_scipy L U: True
Course check, P_course A = L U: True


### 1.2 Forward and Back Substitution

Using the course convention $P_{\rm course}A=LU$, solving $Ax=b$ becomes two triangular solves:
1. Solve $Ly=P_{\rm course}b$ (forward substitution — work top to bottom)
2. Solve $Ux = y$ (back substitution — work bottom to top)

For SciPy's returned matrix, $P_{\rm course}=P^{\top}$, so the first right-hand side below is `P.T @ b`. NumPy's `np.linalg.solve` does this automatically, but we implement it manually to see the steps. This is the computational version of Written HW2 Problems 1 and 2.


In [3]:
# 1.2 Manual forward substitution: solve Ly = b where L is unit lower triangular
def forward_sub(L, b):
    """Solve Ly = b for y, given unit lower-triangular L."""
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - L[i, :i] @ y[:i]
    return y

def back_sub(U, y):
    """Solve Ux = y for x, given upper-triangular U."""
    n = len(y)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x

# Test: solve Ax = b
b = np.array([4., 14., 40.])

# LU solve: forward then back
y = forward_sub(L, P.T @ b)   # scipy: A=PLU, so LUx=P.T@b
x_lu = back_sub(U, y)

# NumPy reference
x_ref = np.linalg.solve(A, b)

print("LU solution:  ", x_lu.round(6))
print("NumPy solution:", x_ref.round(6))
print("Match:", np.allclose(x_lu, x_ref))


LU solution:   [-1.  3.  3.]
NumPy solution: [-1.  3.  3.]
Match: True


**Your turn:** Implement `lu_solve(A, b)` — a function that uses `scipy.linalg.lu` and your `forward_sub`/`back_sub` to solve $Ax = b$.


In [4]:
def lu_solve(A, b):
    A = np.array(A)
    b = np.array(b)
    # Note: scipy returns (P,L,U) with A = P@L@U, so P.T@b is correct
    # Hint: you have forward_sub and back_sub above

    # Factor A using scipy.linalg.lu
    P, L, U = lu(A)

    # Solve Ly = P.T @ b using forward_sub
    y = forward_sub(L, P.T @ b)

    # Solve Ux = y using back_sub
    x = back_sub(U, y)

    return x

# Test
b_test = np.array([1., 2., 3.])
x_test = lu_solve(A, b_test)
assert np.allclose(A @ x_test, b_test), "lu_solve incorrect"
print("lu_solve: all tests passed ✓")


lu_solve: all tests passed ✓


### 1.3 Matrix Inverses — When to Use Them (and When Not To)

`np.linalg.inv(A)` computes $A^{-1}$. It's tempting to solve $Ax = b$ as $x = A^{-1}b$, but **this is almost always a mistake** in practice:
- Computing $A^{-1}$ costs the same as two LU factorizations
- It accumulates more floating-point error than solving directly
- `np.linalg.solve` uses LU internally and is faster and more accurate

**When is `inv` appropriate?** When you genuinely need the inverse matrix itself — not just a solution to one system.


In [5]:
# 1.3 Timing: solve vs. forming an inverse and multiplying
import time

n = 200
rng = np.random.default_rng(412)
A_big = rng.standard_normal((n, n))
A_big = A_big @ A_big.T + n * np.eye(n)   # symmetric positive definite
b_big = rng.standard_normal(n)

def median_runtime(operation, repeats=7):
    """Median wall-clock time for one complete operation."""
    samples = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        operation()
        samples.append(time.perf_counter() - t0)
    return np.median(samples)

# Both timings include all work needed for one right-hand side.
t_solve = median_runtime(lambda: np.linalg.solve(A_big, b_big))
t_inverse = median_runtime(lambda: np.linalg.inv(A_big) @ b_big)

x_solve = np.linalg.solve(A_big, b_big)
x_inverse = np.linalg.inv(A_big) @ b_big
print(f"solve(A, b):      {t_solve*1e3:.2f} ms")
print(f"inv(A) @ b:       {t_inverse*1e3:.2f} ms")
print("Solutions match:", np.allclose(x_solve, x_inverse))
print("Residual from solve:", np.linalg.norm(A_big @ x_solve - b_big))
print("Residual from inverse:", np.linalg.norm(A_big @ x_inverse - b_big))


solve(A, b):      0.25 ms
inv(A) @ b:       0.52 ms
Solutions match: True
Residual from solve: 9.34621743851611e-15
Residual from inverse: 8.072762667609755e-15


### 1.4 Determinants

The determinant $\det(A)$ encodes the signed volume scaling factor of $A$ as a linear map.

Key facts:
- $\det(A) \neq 0$ iff $A$ is invertible
- $\det(AB) = \det(A)\det(B)$
- $\det(A^\top) = \det(A)$
- Row swap flips sign; scaling row $i$ by $c$ scales $\det$ by $c$

After LU factorization $PA = LU$:
$$\det(A) = \det(P)^{-1}\cdot\det(L)\cdot\det(U) = \pm\prod_i u_{ii}$$
since $\det(L) = 1$ (unit lower triangular) and $\det(P) = \pm 1$.


In [6]:
# 1.4 Compute det from LU (far cheaper than cofactor expansion for large matrices)
def det_from_lu(A):
    """Compute det(A) using LU factorization."""
    P, L, U = lu(A)
    # SciPy gives A = P L U. Since det(L)=1 and det(P)=±1,
    # det(A) = det(P) times the product of U's diagonal entries.
    return np.linalg.det(P) * np.prod(np.diag(U))

# Verify
B = np.array([[3., 1.], [5., 2.]])
print(f"det_from_lu(B) = {det_from_lu(B):.6f}")
print(f"np.linalg.det(B) = {np.linalg.det(B):.6f}")
assert np.isclose(det_from_lu(B), np.linalg.det(B))
print("det_from_lu: verified ✓")


det_from_lu(B) = 1.000000
np.linalg.det(B) = 1.000000
det_from_lu: verified ✓


**Your turn:** Write `is_invertible(A, tol=1e-10)` using the determinant test from Unit 7. Here `tol` treats a sufficiently small computed determinant as zero. This is a useful exercise, although in serious numerical work invertibility also has to be judged relative to the matrix's scale and condition number.


In [7]:
def is_invertible(A, tol=1e-10):
    A = np.array(A)
    det = det_from_lu(A)
    return abs(det) > tol
    pass

assert is_invertible(np.eye(3)) == True
assert is_invertible(np.zeros((3, 3))) == False
assert is_invertible(np.array([[1., 2.], [2., 4.]])) == False
print("is_invertible: all tests passed ✓")


is_invertible: all tests passed ✓


---

## Part 2: Data Science / AI Application — Fitting a Linear Model

### 2.1 Linear Regression via Normal Equations (preview)

Suppose we have $m$ data points $(x_i, y_i)$ and want to fit the model:
$$y \approx c_0 + c_1 x + c_2 x^2$$

Stacking all equations: $Ac = y$ where row $i$ of $A$ is $[1,\; x_i,\; x_i^2]$.

This system is overdetermined ($m \gg 3$), so we solve the **normal equations**:
$$A^\top A\, c = A^\top y$$

The matrix $A^\top A$ is $3 \times 3$ and symmetric — solve it with LU. Least squares is developed fully in Unit 23; here it is an application of this week's matrix-solving tools.


In [8]:
# 2.1 Polynomial regression using normal equations
rng = np.random.default_rng(412)
m = 50
x_data = np.linspace(-2, 2, m)
y_data = 1.5 - 0.3*x_data + 0.8*x_data**2 + rng.standard_normal(m) * 0.3

# Build the Vandermonde design matrix
A_design = np.column_stack([np.ones(m), x_data, x_data**2])
print("Design matrix shape:", A_design.shape)

# Solve normal equations: (A^T A) c = A^T y
ATA = A_design.T @ A_design
ATy = A_design.T @ y_data
c = np.linalg.solve(ATA, ATy)

print(f"Fitted coefficients: c0={c[0]:.4f}, c1={c[1]:.4f}, c2={c[2]:.4f}")
print(f"True coefficients:   c0=1.5000, c1=-0.3000, c2=0.8000")

# Check invertibility
P, L, U = lu(ATA)
print(f"Pivots of A^T A: {np.diag(U).round(4)}")   # nonzero here, so the system has a unique solution


Design matrix shape: (50, 3)
Fitted coefficients: c0=1.4311, c1=-0.3330, c2=0.7949
True coefficients:   c0=1.5000, c1=-0.3000, c2=0.8000
Pivots of A^T A: [ 69.3878  69.3878 -55.4436]


### 2.2 Condition Number — How Sensitive Is Your Solution? (preview of Unit 19)

The **condition number** of $A$ measures how much the solution $x = A^{-1}b$ changes when $b$ is perturbed:
$$\text{cond}(A) = \|A\|\cdot\|A^{-1}\|$$

Rule of thumb: if $\text{cond}(A) \approx 10^k$, expect to lose about $k$ digits of precision.


In [9]:
# 2.2 Condition number of the design matrix
cond_ATA = np.linalg.cond(ATA)
print(f"cond(A^T A)      = {cond_ATA:.2e}")
print(f"cond(A_design)   = {np.linalg.cond(A_design):.2e}")

# Experiment: ill-conditioned system
x_ill = np.linspace(0, 0.01, 10)  # tiny range → Vandermonde columns nearly parallel
A_ill = np.column_stack([np.ones(10), x_ill, x_ill**2])
print(f"cond(A_ill)      = {np.linalg.cond(A_ill):.2e}")
print("↑ Very ill-conditioned — solution will be numerically unreliable")


cond(A^T A)      = 1.09e+01
cond(A_design)   = 3.30e+00
cond(A_ill)      = 1.11e+05
↑ Very ill-conditioned — solution will be numerically unreliable


---

## Part 3: 🤖 Mini-Transformer Milestone

### Week 2 Milestone — Diagnosing a Linear Projection

This week we use embeddings from Coding HW1 to study whether a linear projection is uniquely determined. The same issue appears by hand in Written HW2 Problem 4: when feature columns are dependent, different weights can produce the same observed outputs.

Our toy vocabulary:

| Token | ID |
|-------|-----|
| `[BOS]` | 0 |
| `the` | 1 |
| `cat` | 2 |
| `sat` | 3 |
| `mat` | 4 |
| `[EOS]` | 5 |


In [10]:
# Milestone 3.1: Rebuild X from HW 1
# You built the embedding matrix and the lookup in Coding HW 1. One line each here.
import mt_utils

mt_utils.set_seed(412)
VOCAB_SIZE, D_MODEL = 6, 8

E = mt_utils.make_embedding_matrix(VOCAB_SIZE, D_MODEL)
X = mt_utils.embed_tokens([1, 2, 3], E)        # "the cat sat"

print("E:", E.shape, " X:", X.shape)
print("Recall from HW 1: X = P @ E, but nobody computes it that way.")


E: (6, 8)  X: (3, 8)
Recall from HW 1: X = P @ E, but nobody computes it that way.


In [11]:
# Milestone 3.2: Diagnose a rank-deficient projection problem
# Suppose we want W such that X @ W = X_target.
rng = np.random.default_rng(42)
W_true = rng.standard_normal((D_MODEL, D_MODEL)) * 0.1
X_target = X @ W_true

# X has 3 rows and 8 columns, so rank(X) can be at most 3.
XTX = X.T @ X
print(f"X shape: {X.shape}, rank(X): {np.linalg.matrix_rank(X)}")
print(f"X^T X shape: {XTX.shape}, rank(X^T X): {np.linalg.matrix_rank(XTX)}")
print("The 8 x 8 normal-equation matrix is singular, so LU cannot give a unique W.")
print("More observations can make full column rank possible, but do not guarantee it.")
print("As in Written HW2 Problem 4(c), different weights may agree on all observed inputs.")


X shape: (3, 8), rank(X): 3
X^T X shape: (8, 8), rank(X^T X): 3
The 8 x 8 normal-equation matrix is singular, so LU cannot give a unique W.
More observations can make full column rank possible, but do not guarantee it.
As in Written HW2 Problem 4(c), different weights may agree on all observed inputs.


In [12]:
# Milestone 3.3 (TODO): Find one solution with least squares
# Because X has more columns than rows, solve X @ W ≈ X_target directly.
# np.linalg.lstsq returns one minimum-norm solution; it does not need X^T X to be invertible.

W_approx = np.linalg.lstsq(X, X_target, rcond=None)[0]

# Once filled in, this should print a reconstruction error close to zero.
if W_approx is not None:
    err = np.linalg.norm(X_target - X @ W_approx, 'fro')
    print(f"Reconstruction error ||X_target - X W_approx||_F = {err:.6f}")
    print("W_approx equals W_true:", np.allclose(W_approx, W_true))
    print("A tiny reconstruction error with unequal matrices demonstrates non-uniqueness.")


Reconstruction error ||X_target - X W_approx||_F = 0.000000
W_approx equals W_true: False
A tiny reconstruction error with unequal matrices demonstrates non-uniqueness.


### ✅ Milestone Checklist

Before submitting, verify:
- [ ] `lu_solve` and `is_invertible` both pass their assertions
- [ ] Normal equations give fitted coefficients close to the true values
- [ ] `W_approx` gives reconstruction error close to zero, even though it need not equal `W_true`
- [ ] You can connect the non-unique `W` matrices to Written HW2 Problem 4(c)
- [ ] You can say why `np.linalg.solve(A, b)` beats `inv(A) @ b`
